# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

*Loads the decision-point frame from ML-04 (features Feb-Apr 2026, label `y_decline_may` from May 2026). If the cached file is missing, the cell rebuilds it from the warehouse (needs `HF_TOKEN`). June 2026 is never loaded.*

In [1]:
# Setup | load the decision-point frame; build it from the warehouse only if it is not cached (features Feb-Apr 2026, label May 2026; June is never loaded)
import gc, json, os
import numpy as np, pandas as pd

FRAME = "work/outputs/frame_dp_2026-05-01.parquet"
os.makedirs("work/outputs", exist_ok=True)

def hf_token():
    try:
        from google.colab import userdata          # Colab: Secrets panel
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")          # local run: environment variable

if os.path.exists(FRAME):
    print("Using the cached frame:", FRAME)
else:
    # ML-04 v2 | Decision-point frame: features Feb-Apr 2026, label May 2026. June stays sealed (never loaded).
    import gc, json, os
    import numpy as np, pandas as pd

    HF = {"token": hf_token()}
    HF_BASE = "hf://datasets/FlyRank/internship-warehouse"
    FEAT_MONTHS, LABEL_MONTH = ["2026-02", "2026-03", "2026-04"], "2026-05"
    MIN_IMPR_WINDOW, MIN_CLICKS_APR, DROP_RATIO = 1000, 10, 0.80      # contract thresholds: frozen before any model
    KEYS = ["client_hash_id", "content_hash_id"]
    COLS = ["report_date", *KEYS, "gsc_clicks", "gsc_impressions", "gsc_avg_position"]
    assert LABEL_MONTH != "2026-06" and "2026-06" not in FEAT_MONTHS, "June is the sealed test month"

    def month_agg(m):
        """One partition -> one row per (client, content). Loads only needed columns; frees RAM right after."""
        path = f"{HF_BASE}/fact_content_daily_performance/month={m}/"
        try:
            d = pd.read_parquet(path, columns=COLS, storage_options=HF)
        except Exception:
            from huggingface_hub import HfFileSystem
            import pyarrow.parquet as pq
            fs = HfFileSystem(token=HF["token"])
            f = fs.ls(f"datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}", detail=False)[0]
            print("Actual schema:", pq.read_schema(f, filesystem=fs).names); raise
        dates = pd.to_datetime(d["report_date"])
        win = (dates.min().date(), dates.max().date(), len(d))
        impr = d["gsc_impressions"].fillna(0)
        pos = d["gsc_avg_position"].where(d["gsc_avg_position"] > 0)            # 0/negative = no data, not rank 0
        d = d.assign(clicks=d["gsc_clicks"].fillna(0), impr=impr,
                     pos_x_impr=(pos * impr).fillna(0), impr_pos=impr.where(pos.notna(), 0),
                     active=(impr > 0).astype("int16"))
        g = d.groupby(KEYS, observed=True).agg(clicks=("clicks", "sum"), impr=("impr", "sum"),
                                               pos_x_impr=("pos_x_impr", "sum"), impr_pos=("impr_pos", "sum"),
                                               active_days=("active", "sum")).reset_index()
        del d; gc.collect()
        return g, win

    # 1) Aggregate each month separately (daily grain -> content grain), check windows land inside their month
    frames, windows = {}, {}
    for m in FEAT_MONTHS + [LABEL_MONTH]:
        frames[m], windows[m] = month_agg(m)
        assert str(windows[m][0]).startswith(m) and str(windows[m][1]).startswith(m), f"{m} partition has dates outside its month"
        print(f"{m}: daily rows={windows[m][2]:,} | {windows[m][0]} -> {windows[m][1]} | contents={len(frames[m]):,}")

    # 2) Feature frame from Feb-Apr ONLY (nine features, all knowable on 2026-05-01)
    f = None
    for m in FEAT_MONTHS:
        part = frames[m].rename(columns={c: f"{c}_{m}" for c in ["clicks", "impr", "pos_x_impr", "impr_pos", "active_days"]})
        f = part if f is None else f.merge(part, on=KEYS, how="outer")
    num = [c for c in f.columns if c not in KEYS]
    f[num] = f[num].fillna(0)                                             # absent row in a month = no search rows that month
    S = lambda p: sum(f[f"{p}_{m}"] for m in FEAT_MONTHS)
    f["impressions_total"], f["clicks_total"] = S("impr"), S("clicks")
    f["impressions_apr"], f["clicks_apr"], f["clicks_feb"] = f["impr_2026-04"], f["clicks_2026-04"], f["clicks_2026-02"]
    f["momentum_apr_vs_feb"] = np.log1p(f["clicks_apr"]) - np.log1p(f["clicks_feb"])
    f["ctr"] = np.where(f["impressions_total"] > 0, f["clicks_total"] / f["impressions_total"], np.nan)
    f["weighted_position"] = np.where(S("impr_pos") > 0, S("pos_x_impr") / S("impr_pos").replace(0, np.nan), np.nan)  # sum(pos*impr)/sum(impr), not a mean of daily means
    f["active_days"] = S("active_days")
    FEATURES = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb",
                "momentum_apr_vs_feb", "ctr", "weighted_position", "active_days"]
    f = f[KEYS + FEATURES]

    # 3) Label from May ONLY: decline = May clicks < 80% of April clicks. Label ingredients are never features.
    may = frames[LABEL_MONTH][KEYS + ["clicks"]].rename(columns={"clicks": "clicks_may"})
    n_before_eligibility = len(f)
    f = f[(f["impressions_total"] >= MIN_IMPR_WINDOW) & (f["clicks_apr"] >= MIN_CLICKS_APR)]   # eligibility, decided at the decision point
    f = f.merge(may, on=KEYS, how="left")
    missing_in_may = int(f["clicks_may"].isna().sum())
    f["clicks_may"] = f["clicks_may"].fillna(0)                           # not seen in May -> 0 clicks -> counts as decline (stated in contract)
    f["y_decline_may"] = (f["clicks_may"] < DROP_RATIO * f["clicks_apr"]).astype(int)
    LABEL_INGREDIENTS = ["clicks_may"]                                    # kept ONLY to build/audit the label
    frame = f.reset_index(drop=True)

    # 4) Contract checks (queries, not claims)
    assert not frame.duplicated(KEYS).any(), "grain broken: (client, content) must be unique"
    assert not set(LABEL_INGREDIENTS) & set(FEATURES), "label ingredient leaked into features"
    assert set(KEYS).isdisjoint(FEATURES), "IDs are grouping keys, never features"
    share = frame["client_hash_id"].value_counts(normalize=True)
    base_rate = float(frame["y_decline_may"].mean())
    print(f"\nBefore eligibility: {n_before_eligibility:,} | Eligible rows: {len(frame):,} | Clients: {frame['client_hash_id'].nunique()}")
    print(f"Base rate (decline in May): {base_rate:.3f} | Not seen in May (counted as decline): {missing_in_may:,}")
    print(f"Largest client share of rows: {share.iloc[0]:.1%} | Top-5 clients: {share.head(5).sum():.1%}")
    print("Per-client base rate spread (min / median / max, clients with >=30 rows):")
    pc = frame.groupby("client_hash_id")["y_decline_may"].agg(["mean", "size"]); pc = pc[pc["size"] >= 30]["mean"]
    print(f"  {pc.min():.2f} / {pc.median():.2f} / {pc.max():.2f}  over {len(pc)} clients")
    print("Missing in features (share):"); print(frame[FEATURES].isna().mean().round(3).to_string())

    # 5) Receipt (commit this JSON), cache the frame locally (do NOT commit the parquet)
    os.makedirs("work/outputs", exist_ok=True)
    frame.to_parquet("work/outputs/frame_dp_2026-05-01.parquet", index=False)
    json.dump({"decision_point": "2026-05-01", "feature_months": FEAT_MONTHS, "label_month": LABEL_MONTH,
               "min_impr_window": MIN_IMPR_WINDOW, "min_clicks_apr": MIN_CLICKS_APR, "drop_ratio": DROP_RATIO,
               "rows_before_eligibility": int(n_before_eligibility), "rows": int(len(frame)),
               "clients": int(frame["client_hash_id"].nunique()), "base_rate": round(base_rate, 4),
               "not_seen_in_may": missing_in_may, "features": FEATURES, "label": "y_decline_may",
               "group_key": "client_hash_id", "sealed_month": "2026-06", "release": "v20260703"},
              open("work/outputs/data_contract_v2.json", "w"), indent=2)
    print("\nSaved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)")

2026-02: daily rows=7,355,108 | 2026-02-01 -> 2026-02-28 | contents=321,546
2026-03: daily rows=9,841,378 | 2026-03-01 -> 2026-03-31 | contents=331,437
2026-04: daily rows=10,424,730 | 2026-04-01 -> 2026-04-30 | contents=362,172
2026-05: daily rows=11,687,376 | 2026-05-01 -> 2026-05-31 | contents=389,153

Before eligibility: 380,147 | Eligible rows: 16,513 | Clients: 36
Base rate (decline in May): 0.416 | Not seen in May (counted as decline): 0
Largest client share of rows: 26.1% | Top-5 clients: 71.9%
Per-client base rate spread (min / median / max, clients with >=30 rows):
  0.03 / 0.38 / 0.75  over 17 clients
Missing in features (share):
impressions_total      0.0
clicks_total           0.0
impressions_apr        0.0
clicks_apr             0.0
clicks_feb             0.0
momentum_apr_vs_feb    0.0
ctr                    0.0
weighted_position      0.0
active_days            0.0

Saved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)


## 1. Two paper findings + my methodology questions

*Source: "The State of AI-Driven SEO, March 2026" (FlyRank). I am not grading the paper. It says it was built for a broad audience, that its ML pages are exploratory, and that it reports no p-values or confidence intervals (pp. 2 and 36). These are the questions I would want a reviewer to ask about my own work, so I practice them here.*

### Finding 1 - "What predicts health?" (p. 27, with the Health Score recipe on p. 5)

**The claim.** A Random Forest predicting the Health Score leans most on average position (43%), impressions (32%), scroll depth (15%) and CTR (8%).

**Where does the label come from?** From the recipe on p. 5: impressions (30 points) + position (30) + CTR (20) + scroll depth (20). Those are exactly the four features at the top of the chart, and together they hold 98 of the 100 importance points. The paper already says the target is "partly constructed" from these inputs and that importance is descriptive. I would go one step further: with 98 of 100 points on the recipe's own inputs, the chart looks like the recipe being read back, so "partly" could be "almost entirely".

**Does the validation design carry the claim?** The holdout shows the model can rebuild the score for pages it has not seen. That supports "the model can recompute its own recipe", not "these are the drivers of healthy content".

**What would carry a claim about drivers.** Predict an outcome that is *not* built from the inputs (for example clicks in the next 30 days), or remove the recipe inputs and ask what is left.

**What I saw when I tried the idea on my own data.** I built a recipe-style score from three of my features (impressions, position, CTR; scroll depth is not in my data) and trained a forest to predict it. The holdout R2 was 0.997. The importance went to `weighted_position` (64.1%), `clicks_total` (31.7%), `impressions_total` (2.6%) and `ctr` (1.6%), and **0.0% to everything else** (for example `active_days` and `momentum_apr_vs_feb`). `clicks_total` is simply CTR x impressions, so it carries the recipe's information too. Importance gets shared between overlapping columns, but all of it stays inside the recipe's information. This is the same pattern as the paper's chart, produced on purpose.

### Finding 2 - "What predicts growth?" (p. 29, Logistic Regression, 71% holdout accuracy)

**The claim.** A Logistic Regression separates growing from declining pages with 71% holdout accuracy, and "days visible" and "recent impressions" are among the strongest positive signals.

**Where does the label come from?** The page does not define "growing". If it is the trend direction defined on p. 5 (30-day versus previous-30-day impression change, up or down by more than 10%), the label is built from impressions. A "recent impressions" feature might then share a time window with the label. The paper does not say which window the feature covers, so that is my first question.

**Does the validation design carry the claim?** The method page (p. 36) says 80/20 split. It does not say what was split. With 57 brands, the pages of one brand can sit on both sides, which would answer "does it work on a page from a brand it knows?" and not "on a new brand?". The page also gives no base rate, and accuracy only means something next to it. For example, if 62% of the pages share one class, 71% accuracy is 9 points of skill, not 71.

**What would carry the claim.** Report the majority-class accuracy next to 71%, split by brand, make sure the features end before the label window starts, and say how many pages and how many repeated runs stand behind the number.

**What I saw on my own model.** At the usual 0.5 threshold my Logistic Regression has accuracy 0.66, 0.52, 0.61, 0.63 and 0.48 across the five client-grouped folds. Always guessing the majority class gives 0.67, 0.66, 0.60, 0.63 and 0.52. The skill over the majority guess is -0.7, -14.0, +1.0, 0.0 and -3.9 points. Accuracy alone would say the model is no better than a constant guess, yet at the head of the review queue the same model reaches precision@50 of 0.62 against a base rate of 0.39 (ML-08). So accuracy needs the majority-class number beside it, and the metric has to match the use: here, ranking pages at a review budget of 50.

In [2]:
# 1) Shared setup (same rows, folds and rule as ML-08) + two small demonstrations that back my questions about the paper
import json, platform
import numpy as np, pandas as pd, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

SEED, K_MAIN = 42, 50
KEYS, LABEL = ["client_hash_id", "content_hash_id"], "y_decline_may"
FEATURES = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb",
            "momentum_apr_vs_feb", "ctr", "weighted_position", "active_days"]
COUNTS = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb"]
MIN_RATIO, HISTORY_FLOOR, BLOCK_COPIES = 2.0, 10, 20                                  # frozen ML-07 rule settings
VEC = ["clicks_feb", "clicks_mar", "clicks_apr", "impressions_apr", "weighted_position", "active_days"]

def add_derived(f):
    f = f.copy()
    f["clicks_mar"] = (f["clicks_total"] - f["clicks_apr"] - f["clicks_feb"]).clip(lower=0)      # middle month of the feature window
    prior = (f["clicks_feb"] + f["clicks_mar"]) / 2
    f["ratio_apr"] = f["clicks_apr"] / prior.where(prior > 0)
    return f
def drop_block(f):       # identical feature vectors repeated >= 20 times are not independent evidence (decided from features only)
    blk = f.groupby(VEC, dropna=False)["content_hash_id"].transform("size") >= BLOCK_COPIES
    return f[~blk].reset_index(drop=True), int(blk.sum())
def tiebreak(f): return pd.util.hash_pandas_object(f["content_hash_id"].astype(str) + f"|{SEED}", index=False).to_numpy()
def rule_score(f):       # the frozen ML-07 rule
    flag = (f["ratio_apr"] >= MIN_RATIO) & (f["clicks_feb"] >= HISTORY_FLOOR) & (f["clicks_mar"] >= HISTORY_FLOOR)
    strength = (np.log2(f["ratio_apr"].clip(lower=1)) / 3).clip(0, 1)
    return np.where(flag, 100 * strength * f["impressions_apr"].rank(pct=True), 0.0).round(4)
def make_models(feats=FEATURES):      # fixed settings from ML-08, no tuning
    counts, others = [c for c in COUNTS if c in feats], [c for c in feats if c not in COUNTS]
    steps = ([("counts", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), counts)] if counts else []) + [("others", StandardScaler(), others)]
    return {"Logistic Regression": make_pipeline(ColumnTransformer(steps), LogisticRegression(max_iter=2000)),
            "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED)}
def prec_at_k(y_, s_, tb_, k):
    o = np.lexsort((tb_, -s_)); return float(y_[o][:k].mean())

df, n_block = drop_block(add_derived(pd.read_parquet(FRAME)))
df["_tb"], df["rule_score"] = tiebreak(df), rule_score(df)
X, y, groups, tb = df[FEATURES], df[LABEL].to_numpy(), df["client_hash_id"].to_numpy(), df["_tb"].to_numpy()
folds = list(GroupKFold(n_splits=5).split(X, y, groups))                                 # the same five client-grouped folds as ML-08
RULE = "Frozen rule (ML-07)"
print(f"Analysis frame: {len(df):,} rows | {df['client_hash_id'].nunique()} clients | base rate {y.mean():.3f} | excluded identical block: {n_block:,} rows")
print(f"Versions: python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} | seed {SEED}\n")

# Demonstration A (paper finding 1): a model that is asked to predict a score built from some inputs hands the importance back to those inputs.
print("Paper p.27 importance: position 43, impressions 32, scroll depth 15, CTR 8 =", 43 + 32 + 15 + 8, "of 100. Paper p.5 recipe inputs: impressions, position, CTR, scroll depth.")
pct = lambda s: s.rank(pct=True)
health_like = 30 * pct(df["impressions_total"]) + 30 * (1 - pct(df["weighted_position"])) + 20 * pct(df["ctr"])    # same recipe idea, minus scroll depth (not in our data)
Xtr, Xte, ytr, yte = train_test_split(X, health_like, test_size=0.2, random_state=SEED)
rfr = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=SEED).fit(Xtr, ytr)
pi = pd.Series(permutation_importance(rfr, Xte, yte, scoring="r2", n_repeats=3, random_state=SEED, n_jobs=-1).importances_mean, index=FEATURES).clip(lower=0)
share = (100 * pi / pi.sum()).round(1).sort_values(ascending=False)
print(f"On OUR frame: a forest predicting a recipe-built score (holdout R2 = {rfr.score(Xte, yte):.3f}). Importance shares (%):")
print(share.to_string())
rec3 = share[['impressions_total', 'weighted_position', 'ctr']].sum()
print(f"The three recipe inputs hold {rec3:.1f}%; clicks_total (= CTR x impressions, arithmetic on two of them) holds {share['clicks_total']:.1f}% more; everything else {share.drop(['impressions_total', 'weighted_position', 'ctr', 'clicks_total']).sum():.1f}%. The chart reads the recipe back.\n")

# Demonstration B (paper finding 2): accuracy only means something next to the share of the majority class
rows = []
for i, (tr, te) in enumerate(folds, 1):
    m = make_models()["Logistic Regression"].fit(X.iloc[tr], y[tr])
    acc, maj = float((m.predict(X.iloc[te]) == y[te]).mean()), float(max(y[te].mean(), 1 - y[te].mean()))
    rows.append({"fold": i, "decline rate (base rate)": round(float(y[te].mean()), 3), "LR accuracy": round(acc, 3),
                 "accuracy of always guessing the majority": round(maj, 3), "skill over majority (points)": round(100 * (acc - maj), 1)})
print("Accuracy of my own Logistic Regression next to the majority-class guess (client-grouped folds):")
print(pd.DataFrame(rows).to_string(index=False))

Analysis frame: 15,680 rows | 36 clients | base rate 0.385 | excluded identical block: 833 rows
Versions: python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1 | seed 42

Paper p.27 importance: position 43, impressions 32, scroll depth 15, CTR 8 = 98 of 100. Paper p.5 recipe inputs: impressions, position, CTR, scroll depth.
On OUR frame: a forest predicting a recipe-built score (holdout R2 = 0.997). Importance shares (%):
weighted_position      64.1
clicks_total           31.7
impressions_total       2.6
ctr                     1.6
impressions_apr         0.0
clicks_feb              0.0
clicks_apr              0.0
momentum_apr_vs_feb     0.0
active_days             0.0
The three recipe inputs hold 68.3%; clicks_total (= CTR x impressions, arithmetic on two of them) holds 31.7% more; everything else 0.0%. The chart reads the recipe back.

Accuracy of my own Logistic Regression next to the majority-class guess (client-grouped folds):
 fold  decline rate (base rate)  LR accurac

## 2. My model under an honest split (before/after)

**The design ladder.** The same two models (Logistic Regression and the Random Forest) and the frozen ML-07 rule are scored at precision@50 under four increasingly honest designs:

1. **Random rows** (the "before"): pages of the same client are on both sides, so a model can recognize the client.
2. **Client-grouped folds** (the ML-08 "after"): every client is in exactly one test fold.
3. **Time only**: train at the 1 April decision point (features Jan-Mar, label April), test at 1 May without changing anything. The test clients were seen in training. This is one 50-page list, so it carries a chance margin of roughly +/-14 points.
4. **Client + time**: train on April pages of *other* clients, test on May pages of the held-out clients. This crosses both boundaries.

**Precision@50 (mean over folds; the base rate is what random picking gives).**

| Design | Frozen rule | Logistic Regression | Random Forest | Base rate |
|---|---|---|---|---|
| 1. Random rows (before) | 0.37 | 0.58 | **0.73** | 0.385 |
| 2. Client-grouped (ML-08) | 0.36 | 0.62 | 0.50 | 0.391 |
| 3. Time only | 0.42 | 0.58 | 0.50 | 0.385 |
| 4. Client + time | 0.36 | 0.54 | 0.56 | 0.391 |

AUC under the same four designs: rule 0.502 / 0.501 / 0.502 / 0.501, Logistic Regression 0.598 / 0.581 / 0.558 / 0.569, Random Forest 0.628 / 0.571 / 0.565 / 0.559. The spread across folds is 0.08 (LR) and 0.16 (RF) under client-grouped folds, and 0.13 (LR) and 0.07 (RF) under client + time.

**What the before/after shows (observed, directional).**

- **The Random Forest score was inflated by seeing the same clients.** It falls from 0.73 with random rows to 0.50 with client-grouped folds and 0.56 with client + time, so roughly 17 to 23 points of the "before" number came from recognizing clients and not from a pattern that carries over.
- **The Logistic Regression did not inflate:** 0.58 with random rows, 0.62 grouped, 0.58 time-only, 0.54 under the strictest design. A low-capacity model cannot memorize clients, so I read the small differences as noise.
- **Both models stay above random picking and above the rule under the strictest design.** Logistic Regression is 15 points above the base rate and 18 above the rule, the Random Forest 17 and 19 points. Fold by fold under client + time, the Logistic Regression is above the rule in 5 of 5 folds (by only 4 points in two of them) and above the base rate in 4 of 5 (fold 2 by 2 points, fold 5 below it by 2 points).
- **The model order from ML-08 does not hold up.** In ML-08 the Logistic Regression won clearly. Under client + time it scores 0.54 +/- 0.13 against 0.56 +/- 0.07 for the Random Forest, which is no difference at this size. I rewrite that claim in section 4.
- **AUC is about 0.56 to 0.57 under the honest designs** (0.60 to 0.63 with random rows), so the ranking is weak overall.
- **The label rate moved.** The base rate was 0.538 at the April decision point and 0.385 at the May one. The scores order pages, but they are not calibrated probabilities, and calibration was not evaluated.
- **The same block of 833 identical pages (one client) also appears in the April frame** and is excluded there too.

**Limits.** This is one forward month, so it is thin temporal evidence and not a full walk-forward test.

In [3]:
# 2) My model under honest splits: BEFORE (random rows) -> AFTER (client-grouped) -> and a time-forward check
def run_cv(splits, label):
    out = []
    for name in ("Logistic Regression", "Random Forest"):
        for i, (tr, te) in enumerate(splits, 1):
            s = make_models()[name].fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
            out.append({"design": label, "method": name, "fold": i, "base_rate": float(y[te].mean()), "p50": prec_at_k(y[te], s, tb[te], K_MAIN), "auc": roc_auc_score(y[te], s)})
    for i, (tr, te) in enumerate(splits, 1):
        s = df["rule_score"].to_numpy()[te]
        out.append({"design": label, "method": RULE, "fold": i, "base_rate": float(y[te].mean()), "p50": prec_at_k(y[te], s, tb[te], K_MAIN), "auc": roc_auc_score(y[te], s)})
    return out
rand_splits = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y))
cv = pd.DataFrame(run_cv(rand_splits, "random rows") + run_cv(folds, "client-grouped"))
ladder = [{"design": d_, "method": m_, "P@50": g["p50"].mean(), "AUC": g["auc"].mean(), "base rate": g["base_rate"].mean(), "P@50 sd across folds": g["p50"].std()}
          for (d_, m_), g in cv.groupby(["design", "method"], sort=False)]

# Time-forward check: train at the 1 April decision point (features Jan-Mar, label April), test at 1 May WITHOUT retraining or changing anything.
# Feature names keep their ML-04 names but mean "last month of the window" (impressions_apr, clicks_apr) and "first month" (clicks_feb).
tf_ok, FRAME_APR = True, "work/outputs/frame_dp_2026-04-01.parquet"
def agg_month(m):
    d = pd.read_parquet(f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}/", columns=["report_date", *KEYS, "gsc_clicks", "gsc_impressions", "gsc_avg_position"], storage_options={"token": hf_token()})
    dates = pd.to_datetime(d["report_date"]); assert str(dates.min().date()).startswith(m) and str(dates.max().date()).startswith(m), f"{m}: dates outside the month"
    impr = d["gsc_impressions"].fillna(0); pos = d["gsc_avg_position"].where(d["gsc_avg_position"] > 0)
    d = d.assign(clicks=d["gsc_clicks"].fillna(0), impr=impr, pos_x_impr=(pos * impr).fillna(0), impr_pos=impr.where(pos.notna(), 0), active=(impr > 0).astype("int16"))
    return d.groupby(KEYS, observed=True).agg(clicks=("clicks", "sum"), impr=("impr", "sum"), pos_x_impr=("pos_x_impr", "sum"), impr_pos=("impr_pos", "sum"), active_days=("active", "sum")).reset_index()
def build_frame(feat_months, label_month):
    first, last = feat_months[0], feat_months[-1]
    f = None
    for m in feat_months:
        part = agg_month(m).rename(columns={c: f"{c}_{m}" for c in ["clicks", "impr", "pos_x_impr", "impr_pos", "active_days"]})
        f = part if f is None else f.merge(part, on=KEYS, how="outer")
    f = f.fillna(0); S = lambda p: sum(f[f"{p}_{m}"] for m in feat_months)
    out = f[KEYS].copy()
    out["impressions_total"], out["clicks_total"] = S("impr"), S("clicks")
    out["impressions_apr"], out["clicks_apr"], out["clicks_feb"] = f[f"impr_{last}"], f[f"clicks_{last}"], f[f"clicks_{first}"]
    out["momentum_apr_vs_feb"] = np.log1p(out["clicks_apr"]) - np.log1p(out["clicks_feb"])
    out["ctr"] = np.where(out["impressions_total"] > 0, out["clicks_total"] / out["impressions_total"], np.nan)
    out["weighted_position"] = np.where(S("impr_pos") > 0, S("pos_x_impr") / S("impr_pos").replace(0, np.nan), np.nan)
    out["active_days"] = S("active_days")
    out = out[(out["impressions_total"] >= 1000) & (out["clicks_apr"] >= 10)]                 # same eligibility as ML-04
    nxt = agg_month(label_month)[KEYS + ["clicks"]].rename(columns={"clicks": "clicks_next"})
    out = out.merge(nxt, on=KEYS, how="left"); out["clicks_next"] = out["clicks_next"].fillna(0)
    out["y_decline_next"] = (out["clicks_next"] < 0.8 * out["clicks_apr"]).astype(int)
    return out.reset_index(drop=True)
try:
    if not os.path.exists(FRAME_APR):
        build_frame(["2026-01", "2026-02", "2026-03"], "2026-04").to_parquet(FRAME_APR, index=False)
    apr, n_block_apr = drop_block(add_derived(pd.read_parquet(FRAME_APR)))
    Xa, ya = apr[FEATURES], apr["y_decline_next"].to_numpy()
    print(f"1 April decision point: {len(apr):,} pages | {apr['client_hash_id'].nunique()} clients | base rate {ya.mean():.3f} | identical block excluded: {n_block_apr:,}")
except Exception as e:
    tf_ok = False
    print("TIME-FORWARD CHECK SKIPPED:", type(e).__name__, str(e)[:200])

tf_rows = []
if tf_ok:
    for name in ("Logistic Regression", "Random Forest"):
        s = make_models()[name].fit(Xa, ya).predict_proba(X)[:, 1]                           # (i) time only: all April pages -> all May pages
        tf_rows.append({"design": "time only (train 1 Apr, test 1 May)", "method": name, "fold": 0, "base_rate": float(y.mean()), "p50": prec_at_k(y, s, tb, K_MAIN), "auc": roc_auc_score(y, s)})
    s = df["rule_score"].to_numpy(); tf_rows.append({"design": "time only (train 1 Apr, test 1 May)", "method": RULE, "fold": 0, "base_rate": float(y.mean()), "p50": prec_at_k(y, s, tb, K_MAIN), "auc": roc_auc_score(y, s)})
    for i, (tr, te) in enumerate(folds, 1):                                                  # (ii) client AND time: April pages of OTHER clients -> May pages of the test clients
        keep = ~apr["client_hash_id"].isin(set(groups[te])).to_numpy()
        for name in ("Logistic Regression", "Random Forest"):
            s = make_models()[name].fit(Xa[keep], ya[keep]).predict_proba(X.iloc[te])[:, 1]
            tf_rows.append({"design": "client + time (unseen clients, later month)", "method": name, "fold": i, "base_rate": float(y[te].mean()), "p50": prec_at_k(y[te], s, tb[te], K_MAIN), "auc": roc_auc_score(y[te], s)})
        s = df["rule_score"].to_numpy()[te]
        tf_rows.append({"design": "client + time (unseen clients, later month)", "method": RULE, "fold": i, "base_rate": float(y[te].mean()), "p50": prec_at_k(y[te], s, tb[te], K_MAIN), "auc": roc_auc_score(y[te], s)})
    tf = pd.DataFrame(tf_rows)
    ladder += [{"design": d_, "method": m_, "P@50": g["p50"].mean(), "AUC": g["auc"].mean(), "base rate": g["base_rate"].mean(), "P@50 sd across folds": g["p50"].std() if len(g) > 1 else float("nan")}
               for (d_, m_), g in tf.groupby(["design", "method"], sort=False)]
    print("\nPer-fold precision@50 under the client + time design (test = May pages of unseen clients):")
    print(tf[tf["design"].str.startswith("client")].pivot(index="fold", columns="method", values="p50").round(2).assign(base_rate=lambda t: t.index.map(lambda i: round(float(y[folds[i - 1][1]].mean()), 2))).to_string())
ladder = pd.DataFrame(ladder)
print("\nBEFORE -> AFTER: precision@50 and AUC by validation design (mean over folds; base rate = what random picking gives)")
print(ladder.round(3).to_string(index=False))

1 April decision point: 16,778 pages | 29 clients | base rate 0.538 | identical block excluded: 833

Per-fold precision@50 under the client + time design (test = May pages of unseen clients):
method  Frozen rule (ML-07)  Logistic Regression  Random Forest  base_rate
fold                                                                      
1                      0.24                 0.52           0.52       0.33
2                      0.32                 0.36           0.48       0.34
3                      0.42                 0.62           0.58       0.40
4                      0.38                 0.70           0.66       0.37
5                      0.46                 0.50           0.54       0.52

BEFORE -> AFTER: precision@50 and AUC by validation design (mean over folds; base rate = what random picking gives)
                                     design              method  P@50   AUC  base rate  P@50 sd across folds
                                random rows Logistic Regr

## 3. Leakage audit

**The hunt from Week 3, on the final feature set** (three ways an answer sneaks in):

1. **Label-derived columns.** `clicks_may` builds the label and is never a feature (asserted in code).
2. **Future or overlapping windows.** All features come from February to April, the label from May, and June is never loaded.
3. **Product flags.** No trend direction, health score or optimization flag was ever read. Only clicks, impressions and position were loaded.

The population (impressions and April-click floors) and the identical-row rule are decided from feature columns only, and the code asserts the floors. All of these checks passed.

**The harness.** I added a column built from `clicks_may` on purpose. Precision@50 went to **1.00** and AUC to **1.000**, against 0.62 and 0.58 for the honest model. So the audit can see a leak. The column was removed and is never used.

**Feature-group ablation (client-grouped folds; base rate 0.391).**

| Features | LR P@50 | LR AUC | RF P@50 | RF AUC |
|---|---|---|---|---|
| all 9 features | 0.62 | 0.581 | 0.50 | 0.571 |
| without the 5 click-level features | 0.50 | 0.549 | 0.45 | 0.548 |
| click-level features only | 0.38 | 0.519 | 0.47 | 0.550 |

The label is "May clicks below 80% of April clicks", so the click-level features (`clicks_total`, `clicks_apr`, `clicks_feb`, `momentum_apr_vs_feb`, `ctr`) are close cousins of the label. They are legal inputs, because they are known before May, but the result leans on them: without them the Logistic Regression drops from 0.62 to 0.50 and its AUC from 0.581 to 0.549 (the Random Forest from 0.50 to 0.45). It stays above the base rate, by about 11 points. Click-level features alone do not work (0.38 is the base rate), so the signal comes from clicks and impressions together, which agrees with the coefficient contrast in ML-08. With a spread across folds of about 0.08, a 0.12 gap is a suggestion and not a proof.

**Small-page check** (out-of-fold Logistic Regression scores, client-grouped). In ML-08 I suspected that the gains come from small pages near the 10-click floor.

| April clicks | Pages | Base rate | AUC inside the band | Precision in the top 20% of scores | Lift (points) |
|---|---|---|---|---|---|
| 10-19 | 7,066 | 0.410 | 0.527 | 0.422 | +1.2 |
| 20-49 | 5,635 | 0.378 | 0.532 | 0.428 | +5.0 |
| 50 or more | 2,979 | 0.342 | 0.479 | 0.337 | -0.4 |

Two things stand out. First, the decline rate falls as pages get bigger (41%, 38%, 34%), so small pages decline more often. That fits the regression-to-the-mean story. Second, inside each size band the broad ranking adds little (AUC 0.48 to 0.53, lift between -0.4 and +5.0 points), so a good part of the overall ordering is sorting by page size. **This check looks at the top 20% of scores, not at the top 50.** The top-50 lists hold too few pages per band to read, so whether those 50 pages sit near the floor is still untested.

In [4]:
# 3) Leakage audit on the final feature set
# (1) Timeline and population: features end before the label window; eligibility and the identical-row rule use feature columns only
assert "clicks_may" not in FEATURES and LABEL not in FEATURES and set(KEYS).isdisjoint(FEATURES)
assert (df["impressions_total"] >= 1000).all() and (df["clicks_apr"] >= 10).all()
print("Timeline: features = Feb-Apr 2026 | label = May 2026 | June 2026 never loaded.")
print("Features used:", FEATURES, "\nNo product flags (trend direction, health score, optimization flags) are in the frame: only clicks, impressions and position were read.\n")

# (2) Train WITH and WITHOUT groups of features (client-grouped folds); a collapse would confess dependence on a suspect group
def grouped_eval(feats, data, name="Logistic Regression", return_oof=False):
    ps, au, bs, oof_ = [], [], [], np.full(len(data), np.nan)
    for tr, te in folds:
        s = make_models(feats)[name].fit(data[feats].iloc[tr], y[tr]).predict_proba(data[feats].iloc[te])[:, 1]; oof_[te] = s
        ps.append(prec_at_k(y[te], s, tb[te], K_MAIN)); au.append(roc_auc_score(y[te], s)); bs.append(y[te].mean())
    r = {"P@50": np.mean(ps), "AUC": np.mean(au), "base rate": np.mean(bs)}
    return (r, oof_) if return_oof else r
CLICKY = ["clicks_total", "clicks_apr", "clicks_feb", "momentum_apr_vs_feb", "ctr"]
variants = {"all 9 features": FEATURES, "without click-level features (5 removed)": [c for c in FEATURES if c not in CLICKY], "click-level features only": CLICKY}
rows = [{"variant": v, "model": n, **grouped_eval(fs, df, n)} for v, fs in variants.items() for n in ("Logistic Regression", "Random Forest")]
print("Feature-group ablation, client-grouped folds:"); print(pd.DataFrame(rows).round(3).to_string(index=False))

# (3) Deliberate leak (the harness): add a column built from the label ingredient and watch the score jump, then drop it
leaky = df.assign(leak_may_over_apr=df["clicks_may"] / df["clicks_apr"])
lk = grouped_eval(FEATURES + ["leak_may_over_apr"], leaky)
print(f"\nLeak harness (a column built from clicks_may added on purpose): P@50 = {lk['P@50']:.2f}, AUC = {lk['AUC']:.3f}. The harness works; the column is removed and never used.")

# (4) Small-page check: does the ranking still help once April clicks are not near the eligibility floor of 10?
r_all, oof_lr = grouped_eval(FEATURES, df, return_oof=True)
bins = pd.cut(df["clicks_apr"], [10, 20, 50, np.inf], right=False, labels=["10-19 April clicks", "20-49", "50 or more"])
strata = []
for b in bins.cat.categories:
    idx = np.flatnonzero((bins == b).to_numpy()); yy, ss = y[idx], oof_lr[idx]; top = ss >= np.quantile(ss, 0.8)
    strata.append({"April clicks": b, "pages": len(idx), "base rate": round(float(yy.mean()), 3), "AUC within stratum": round(float(roc_auc_score(yy, ss)), 3),
                   "precision in top 20% of scores": round(float(yy[top].mean()), 3), "lift (points)": round(100 * float(yy[top].mean() - yy.mean()), 1)})
print("\nBy size of the page (out-of-fold Logistic Regression scores, client-grouped):"); print(pd.DataFrame(strata).to_string(index=False))

Timeline: features = Feb-Apr 2026 | label = May 2026 | June 2026 never loaded.
Features used: ['impressions_total', 'clicks_total', 'impressions_apr', 'clicks_apr', 'clicks_feb', 'momentum_apr_vs_feb', 'ctr', 'weighted_position', 'active_days'] 
No product flags (trend direction, health score, optimization flags) are in the frame: only clicks, impressions and position were read.

Feature-group ablation, client-grouped folds:
                                 variant               model  P@50   AUC  base rate
                          all 9 features Logistic Regression 0.616 0.581      0.391
                          all 9 features       Random Forest 0.504 0.571      0.391
without click-level features (5 removed) Logistic Regression 0.500 0.549      0.391
without click-level features (5 removed)       Random Forest 0.452 0.548      0.391
               click-level features only Logistic Regression 0.384 0.519      0.391
               click-level features only       Random Forest 0.472 

## 4. Claim rewrite

**My boldest sentence (what the first, random-split run would have let me write):**

> "My Random Forest finds declining pages with 73% precision at the top 50, 36 points better than the old rule, so it is ready to tell editors which pages to refresh."

**What is wrong with it.**

- The 0.73 was measured with random rows. With client-grouped folds the same model scores 0.50, and with client + time 0.56.
- The gap to the rule is 36 points with random rows, but 14 to 19 points under the honest designs.
- "Tell editors which pages to refresh" is an action claim. The label is an observed proxy (May clicks below 80% of April), and nothing here tests whether refreshing a page helps.
- "Ready" ignores a weak AUC (about 0.57), a spread of 7 to 13 points across folds, a base rate that moved from 0.54 to 0.39 between two months, no calibration, and only one forward month.

**The rewrite (observed, measured, directional, decision-support).**

> "Observed on one decision point (1 May 2026; 15,680 pages from 36 clients): when trained on an earlier month and scored on clients it had not seen, a Logistic Regression and a Random Forest filled a 50-page review list with pages that declined in the following month 54% and 56% of the time, against 39% for random picking and 36% for the frozen rule (spread across folds about 7 to 13 points). The direction held under the stricter designs, but the overall ordering is weak (AUC about 0.57), part of the result depends on click-level features that sit close to the label, the broad ranking adds little inside page-size bands, and the base rate moved from 54% in April to 39% in May. This is decision-support for ordering a human review. It is not evidence that a refresh would help."

**Other claims from ML-08 that I change.**

| Earlier claim | What the audit shows | Safer wording |
|---|---|---|
| "Logistic Regression wins at every K and is the model to carry forward." | Under client + time, LR 0.54 and RF 0.56 are indistinguishable. | "Both models beat the rule and the base rate; neither is clearly better." |
| "A simple model ranks the head of the queue clearly better than the rule and than random." | Above the rule in 5 of 5 folds but by only 4 points in two folds, and below the base rate in one fold. | "Directionally better on average; not stable fold by fold." |
| "The models find small noisy pages." | Small pages do decline more (41% against 34%), but inside size bands the broad lift is small. The top 50 is untested. | "Consistent with a small-page effect at the broad level; the top-50 composition is the next test." |
| "The models lean on `active_days` and the impressions contrast." | Removing the click-level features cuts the Logistic Regression from 0.62 to 0.50. | "The result depends partly on click-level features that are close to the label." |

**What I would test next.** The size-band mix of the top-50 lists, more forward windows (a walk-forward over several months), calibration of the scores, and a per-client view. None of this is claimed here.

In [5]:
# 4) Evidence table behind the rewritten claim + receipt
keep = ladder[ladder["method"].isin(["Logistic Regression", RULE])]
print("Evidence for the claim (precision@50; base rate = random picking):")
print(keep.assign(**{c: keep[c].round(3) for c in ["P@50", "AUC", "base rate", "P@50 sd across folds"]}).to_string(index=False))
json.dump({"seed": SEED, "k": K_MAIN, "rows": int(len(df)), "time_forward_ran": bool(tf_ok),
           "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__},
           "design_ladder": ladder.round(4).to_dict("records"),
           "feature_ablation": pd.DataFrame(rows).round(4).to_dict("records"),
           "leak_harness": {k: round(float(v), 4) for k, v in lk.items()}, "size_strata": strata},
          open("work/outputs/w06_validation_metrics.json", "w"), indent=2, default=str)
print("\nSaved: work/outputs/w06_validation_metrics.json (commit this one)")



Evidence for the claim (precision@50; base rate = random picking):
                                     design              method  P@50   AUC  base rate  P@50 sd across folds
                                random rows Logistic Regression 0.580 0.598      0.385                 0.047
                                random rows Frozen rule (ML-07) 0.368 0.502      0.385                 0.023
                             client-grouped Logistic Regression 0.616 0.581      0.391                 0.078
                             client-grouped Frozen rule (ML-07) 0.364 0.501      0.391                 0.086
        time only (train 1 Apr, test 1 May) Logistic Regression 0.580 0.558      0.385                   NaN
        time only (train 1 Apr, test 1 May) Frozen rule (ML-07) 0.420 0.502      0.385                   NaN
client + time (unseen clients, later month) Logistic Regression 0.540 0.569      0.391                 0.129
client + time (unseen clients, later month) Frozen rule (ML-0

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.